# Experiment: Keypoints Time Series Analysis

What this notebook teaches:
- Load canonical CSV exports and rebuild frame-level keypoint objects.
- Compute knee, elbow, and hip angle trajectories.
- Derive smooth curves, angular velocity, and compact statistical features.
- Produce interpretable plots for downstream posture/gesture modeling.


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/sumeyye-agac/human-pose-estimation-experiments.git"
REPO_DIR = Path("/content/human-pose-estimation-experiments")

if "google.colab" in sys.modules:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    raise RuntimeError("Run this notebook from the repository root.")

if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

print(f"Using repo root: {repo_root}")


In [ ]:
def pip_install(*packages: str) -> None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

pip_install("numpy", "pandas", "matplotlib", "scipy")


In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from posebench.features import compute_angular_velocity, extract_joint_angles, smooth_series, summarize_series_features
from posebench.keypoints_schema import CANONICAL_KEYPOINTS

csv_path = repo_root / "results" / "mediapipe_sequence_canonical.csv"
if not csv_path.exists():
    raise FileNotFoundError(
        "Expected results/mediapipe_sequence_canonical.csv. "
        "Run MediaPipe/02_mediapipe_export_and_features.ipynb first."
    )

df = pd.read_csv(csv_path)
print(df.head(2))


In [ ]:
frames = []
for _, row in df.iterrows():
    keypoints = {}
    for name in CANONICAL_KEYPOINTS:
        keypoints[name] = {
            "x": row.get(f"{name}_x"),
            "y": row.get(f"{name}_y"),
            "confidence": row.get(f"{name}_confidence", 0.0),
        }
    frames.append(
        {
            "frame_index": int(row["frame_index"]),
            "timestamp_ms": float(row["timestamp_ms"]),
            "keypoints": keypoints,
        }
    )

joint_triplets = {
    "left_knee_angle": ("left_hip", "left_knee", "left_ankle"),
    "right_elbow_angle": ("right_shoulder", "right_elbow", "right_wrist"),
    "left_hip_angle": ("left_shoulder", "left_hip", "left_knee"),
}
angles_df = extract_joint_angles(frames, joint_triplets, min_confidence=0.2)
angles_df.head()


In [ ]:
fps = 30.0
angles_df["left_knee_smooth"] = smooth_series(angles_df["left_knee_angle"], method="savgol")
angles_df["left_knee_velocity"] = compute_angular_velocity(angles_df["left_knee_smooth"], fps=fps)

plt.figure(figsize=(10, 4))
plt.plot(angles_df["frame_index"], angles_df["left_knee_angle"], alpha=0.45, label="left_knee_angle")
plt.plot(angles_df["frame_index"], angles_df["left_knee_smooth"], linewidth=2, label="left_knee_smooth")
plt.xlabel("Frame")
plt.ylabel("Angle (deg)")
plt.title("Left knee angle over time")
plt.legend()
plt.show()

plt.figure(figsize=(10, 3))
plt.plot(angles_df["frame_index"], angles_df["left_knee_velocity"], label="left_knee_velocity")
plt.xlabel("Frame")
plt.ylabel("deg/s")
plt.title("Left knee angular velocity")
plt.legend()
plt.show()


In [ ]:
feature_summary = {
    column: summarize_series_features(angles_df[column].to_numpy())
    for column in ["left_knee_angle", "right_elbow_angle", "left_hip_angle"]
}
summary_path = repo_root / "results" / "timeseries_feature_summary.json"
summary_path.write_text(json.dumps(feature_summary, indent=2) + "\n", encoding="utf-8")
feature_summary


From these trajectories and summary features, a downstream gesture or posture classifier can be built with classical ML or lightweight sequence models.
